# Scientifically-Accurate Animations with AI + Manim

**MRS Spring 2026 — Tutorial MT01: Deploying Agentic AI in Materials Characterization Workflows**

So far in this tutorial we've used LLMs to **read** (extraction, RAG),
**reason** (agents calling tools), and **structure** (JSON, schemas). This
notebook shows a different mode: **the LLM as a code generator for visual
explanation**.

The pitch is simple. [Manim](https://www.manim.community/) — the open-source
animation library that powers 3Blue1Brown's videos — produces high-quality
mathematical animations from declarative Python code. But writing that
Python takes time and patience. A modern LLM can write it for you, *and*
it knows the underlying physics.

So you describe the concept ("animate Bragg's law with two parallel atomic
planes and an incoming X-ray"), the model writes the Manim scene, you
render it, and you get a publication-quality scientific animation in a few
minutes — with the geometry, the math, and the labels all correct.

## What you'll see

Four animations, each generated from a one-paragraph English prompt:

1. **Bragg's law** — the foundational diffraction condition.
2. **The Ewald sphere** — the geometric construction every textbook
   tries to explain and few succeed.
3. **BaTiO3 cubic→tetragonal phase transition** — unit cell distortion
   linked to the (200) XRD peak splitting in real time.
4. **Ostwald ripening** — a population of small particles coarsens into
   a few large ones via LSW-theory dynamics.

For each one we show the **prompt** (what you'd type into Claude),
**Claude's generated Manim code**, and the **rendered MP4**. Pre-rendered
so you can read along even without LaTeX/ffmpeg installed.

## Why "AI-generated" matters

You could absolutely write this Manim code yourself. The interesting claim
is that **Claude already knows the physics** — Bragg's law, the Ewald
construction, LSW theory, the Curie temperature of BaTiO3 — so the
human-language prompts can stay short and conceptual. You're not babysitting
the math; you're directing.

## Setup (only needed if you want to re-render)

Pre-rendered MP4s ship with this notebook in `../manim_demos/videos/`. To
*regenerate* them on your own machine you need Manim plus its system
dependencies:

```bash
# System deps (Linux)
sudo apt install ffmpeg libcairo2-dev libpango1.0-dev texlive-full
# macOS
brew install ffmpeg cairo pango py3cairo
brew install --cask mactex

# Python
pip install manim
```

Then re-render any scene with:

```bash
cd manim_demos
manim -ql 01_bragg.py Bragg
```

`-ql` = low quality (480p, fast). Use `-qm` (720p) or `-qh` (1080p) for
production output. The 1.8 MB of MP4s in this folder were all rendered at
`-ql` in well under a minute total.

---
## Example 1 — Bragg's Law

The textbook condition $n\lambda = 2d\sin\theta$. We want to see two
parallel atomic planes, two parallel incoming X-rays, the reflected rays,
and explicitly the path-length difference that interferes constructively
when the Bragg condition is satisfied.

### The prompt

> Write a Manim CE scene called `Bragg` that animates Bragg's law of X-ray
> diffraction. Show two parallel atomic planes spaced $d$ apart, with seven
> atoms on each plane drawn as blue dots. Draw two parallel incoming
> X-rays at the n=1 Bragg angle (compute it from $d$ and $\lambda$),
> reflecting off the top and bottom planes. Highlight in red the extra
> path length $2d\sin\theta$ that the lower ray travels — this is the
> path-length difference that gives constructive interference. Label
> $\theta$ at the angle, label $d$ between the planes, and end by
> displaying the Bragg condition $n\lambda = 2d\sin\theta$ along with
> the numerical Bragg angle. Use $d = 2.0$ scene units and $\lambda = 1.5$.

### Claude's code

In [ ]:
"""Bragg's law animation.

Two parallel atomic planes spaced d apart. Two parallel X-rays incident at
angle theta scatter elastically off the planes. The lower ray travels an
extra path length 2 d sin(theta). When that equals n*lambda, the rays
interfere constructively (Bragg condition) and a diffraction peak appears.

Numerical setup uses d = 2.0 scene units, lambda = 1.5 scene units (Cu Kα
analog), so the n=1 Bragg angle is asin(0.375) = 22.0 degrees.
"""
from manim import *
import numpy as np


D_SPACING = 2.0          # spacing between planes (scene units)
WAVELENGTH = 1.5         # X-ray wavelength (scene units)
N_ATOMS = 7              # atoms per plane


class Bragg(Scene):
    def construct(self):
        # ---------- Title ----------
        title = Text("Bragg's Law of X-ray Diffraction", font_size=32).to_edge(UP)
        self.play(Write(title))

        # ---------- Two atomic planes ----------
        plane_y_top = 0.5
        plane_y_bot = plane_y_top - D_SPACING
        x_positions = np.linspace(-4, 4, N_ATOMS)

        atoms_top = VGroup(*[
            Dot([x, plane_y_top, 0], radius=0.10, color=BLUE)
            for x in x_positions
        ])
        atoms_bot = VGroup(*[
            Dot([x, plane_y_bot, 0], radius=0.10, color=BLUE)
            for x in x_positions
        ])
        line_top = Line([-5, plane_y_top, 0], [5, plane_y_top, 0],
                        color=GREY, stroke_width=1).set_opacity(0.4)
        line_bot = Line([-5, plane_y_bot, 0], [5, plane_y_bot, 0],
                        color=GREY, stroke_width=1).set_opacity(0.4)

        # d-spacing label
        d_brace = BraceBetweenPoints([4.5, plane_y_top, 0],
                                     [4.5, plane_y_bot, 0], direction=RIGHT)
        d_label = MathTex("d", font_size=36).next_to(d_brace, RIGHT, buff=0.1)

        self.play(Create(line_top), Create(line_bot),
                  FadeIn(atoms_top), FadeIn(atoms_bot))
        self.play(GrowFromCenter(d_brace), Write(d_label))

        # ---------- Bragg geometry at the n=1 Bragg angle ----------
        theta = np.arcsin(WAVELENGTH / (2 * D_SPACING))   # n=1 condition
        theta_deg = np.degrees(theta)

        # Reflection points on each plane (chosen so rays are parallel & visible)
        x_top = -1.0
        x_bot = x_top + D_SPACING / np.tan(theta)         # geometry: along beam direction
        ref_top = np.array([x_top, plane_y_top, 0])
        ref_bot = np.array([x_bot, plane_y_bot, 0])

        # Direction vectors
        in_dir  = np.array([np.cos(theta), -np.sin(theta), 0])    # incoming (left-to-right, downward)
        out_dir = np.array([np.cos(theta),  np.sin(theta), 0])    # specular reflection (upward)

        # Far ends of the rays
        ray1_in  = Arrow(ref_top - 4 * in_dir,  ref_top, color=YELLOW, buff=0,
                         stroke_width=4, max_tip_length_to_length_ratio=0.05)
        ray1_out = Arrow(ref_top, ref_top + 4 * out_dir, color=YELLOW, buff=0,
                         stroke_width=4, max_tip_length_to_length_ratio=0.05)
        ray2_in  = Arrow(ref_bot - 4 * in_dir,  ref_bot, color=ORANGE, buff=0,
                         stroke_width=4, max_tip_length_to_length_ratio=0.05)
        ray2_out = Arrow(ref_bot, ref_bot + 4 * out_dir, color=ORANGE, buff=0,
                         stroke_width=4, max_tip_length_to_length_ratio=0.05)

        # Theta angle markers between the planes and the rays
        angle_in  = Angle(line_top, Line(ref_top, ref_top - 2 * in_dir),
                          radius=0.5, other_angle=False, color=WHITE)
        angle_in_label = MathTex(r"\theta", font_size=28).move_to(
            ref_top + 0.7 * (np.array([-np.cos(theta/2), -np.sin(theta/2), 0])))
        # Position the label inside the angle wedge
        angle_in_label.move_to(ref_top + 0.85 * np.array([-np.cos(theta/2),
                                                          np.sin(-theta/2), 0]))

        self.play(GrowArrow(ray1_in), GrowArrow(ray2_in))
        self.play(Create(angle_in), Write(angle_in_label))
        self.wait(0.3)
        self.play(GrowArrow(ray1_out), GrowArrow(ray2_out))

        # ---------- Highlight the path-length difference ----------
        # The extra path = 2 d sin(theta).
        # Geometrically: from ref_top, drop perpendiculars onto ray2_in and ray2_out
        # at the start and end of the lower ray's "extra" portion.
        # Project ref_top onto ray2_in:
        v_in_perp_start = ref_bot + np.dot(ref_top - ref_bot, in_dir) * in_dir
        # Project ref_top onto ray2_out:
        v_out_perp_end  = ref_bot + np.dot(ref_top - ref_bot, out_dir) * out_dir

        # The extra distance is from v_in_perp_start to ref_bot,
        # plus from ref_bot to v_out_perp_end. Each = d sin(theta).
        extra_in  = Line(v_in_perp_start, ref_bot, color=RED, stroke_width=8)
        extra_out = Line(ref_bot, v_out_perp_end, color=RED, stroke_width=8)

        # Faint perpendicular construction lines from ref_top to each foot
        perp_in  = DashedLine(ref_top, v_in_perp_start, color=WHITE,
                              stroke_width=2).set_opacity(0.6)
        perp_out = DashedLine(ref_top, v_out_perp_end, color=WHITE,
                              stroke_width=2).set_opacity(0.6)

        self.play(Create(perp_in), Create(perp_out))
        self.play(Create(extra_in), Create(extra_out))

        path_diff_label = MathTex(r"\text{extra path} = 2 d \sin\theta",
                                  font_size=32, color=RED).to_edge(DOWN, buff=1.0)
        self.play(Write(path_diff_label))
        self.wait(1.0)

        # ---------- Bragg condition ----------
        bragg = MathTex(r"n\,\lambda = 2 d \sin\theta",
                        font_size=44).to_edge(DOWN, buff=0.3)
        self.play(Transform(path_diff_label, bragg))
        self.wait(1.5)

        # ---------- Show the actual angle ----------
        theta_value = MathTex(
            rf"\theta = \arcsin\!\left(\frac{{n\lambda}}{{2d}}\right) = "
            rf"{theta_deg:.1f}^\circ",
            font_size=32).next_to(title, DOWN, buff=0.2)
        self.play(Write(theta_value))
        self.wait(2.0)

### The rendered animation

<video controls width="600" muted loop>
  <source src="../manim_demos/videos/01_bragg.mp4" type="video/mp4">
  Your viewer does not support embedded video. Open the file at
  <code>../manim_demos/videos/01_bragg.mp4</code> directly.
</video>

*Bragg's law: incident rays at the Bragg angle interfere constructively because the lower ray's extra path equals nλ.*

**What to look for:** the extra path is highlighted in two red
segments — one from each perpendicular dropped from the upper reflection
point to the lower ray. That's the classical geometric proof of $2d\sin\theta$
in one frame.

---
## Example 2 — The Ewald Sphere

Every diffraction textbook attempts the Ewald construction; most readers
emerge confused. The animation makes it clear: **a reciprocal-lattice point
diffracts when, and only when, it lies on the Ewald sphere**. As the crystal
rotates, points sweep across the sphere and "light up" momentarily.

We use a 2D version (the Ewald *circle*) for clarity — the 3D generalisation
is the same idea with one more dimension.

### The prompt

> Write a Manim CE scene called `Ewald` that animates the 2D Ewald-sphere
> construction. Place a reciprocal lattice as a 7×7 grid of blue dots
> centered on the origin O. Draw the Ewald circle of radius $R = 1/\lambda$
> centered at $C = (-R, 0)$, so the direct beam wavevector $\vec{k}_{\rm in}$
> goes from $C$ to $O$. Highlight that the diffraction condition is *a
> reciprocal-lattice point lies on the circle*. Then rotate the lattice
> through $2\pi$ around the origin. Whenever a lattice point comes near
> the Ewald circle, light it up in red and draw the corresponding
> $\vec{k}_{\rm out}$ arrow from $C$ to that point. Use $R = 2.0$.

### Claude's code

In [ ]:
"""Ewald sphere construction (in 2D — the Ewald CIRCLE for clarity).

Setup:
  - Reciprocal lattice: integer-spaced grid of points centered on origin O.
  - Ewald circle: radius R = 1/lambda, centered at C = -R*x_hat (so the
    direct beam exits the sphere at the origin O = (0,0)).
  - The diffraction condition is that a reciprocal-lattice point lies
    exactly on the circle. As the crystal rotates, lattice points sweep
    across the circle, and at the instant a point touches the circle,
    that reflection diffracts.

We use lambda such that R = 2.0 in scene units, so the sphere encloses
several lattice points and the geometry is clearly visible.
"""
from manim import *
import numpy as np


R = 2.0                    # Ewald circle radius (= 1/lambda in scene units)
LATTICE_RANGE = 3          # reciprocal-lattice points in each direction
DOT_RADIUS = 0.07
ROTATIONS = 2 * PI         # how far to rotate the crystal
RUN_TIME = 8.0


class Ewald(Scene):
    def construct(self):
        # ---------- Title ----------
        title = Text("Ewald Sphere Construction (2D view)",
                     font_size=30).to_edge(UP)
        self.play(Write(title))

        # ---------- Reciprocal lattice ----------
        lattice_points = VGroup(*[
            Dot([h, k, 0], radius=DOT_RADIUS, color=BLUE)
            for h in range(-LATTICE_RANGE, LATTICE_RANGE + 1)
            for k in range(-LATTICE_RANGE, LATTICE_RANGE + 1)
        ])
        # The (0,0) point — direct beam — gets a special color
        for d in lattice_points:
            if np.allclose(d.get_center()[:2], [0, 0]):
                d.set_color(WHITE)
                d.scale(1.3)

        lattice_label = Text("Reciprocal lattice", font_size=20,
                             color=BLUE).to_corner(UR).shift(0.3 * DOWN)

        # ---------- Ewald circle ----------
        center_C = np.array([-R, 0, 0])
        ewald = Circle(radius=R, color=YELLOW).move_to(center_C)
        center_dot = Dot(center_C, radius=0.06, color=YELLOW)
        center_label = MathTex("C", font_size=24,
                               color=YELLOW).next_to(center_dot, LEFT, buff=0.1)

        # ---------- Incoming wavevector k_in: from C to origin O ----------
        k_in = Arrow(center_C, ORIGIN, color=GREEN, buff=0,
                     stroke_width=4, max_tip_length_to_length_ratio=0.08)
        k_in_label = MathTex(r"\vec{k}_{\rm in}", font_size=28,
                             color=GREEN).next_to(k_in.get_center(), DOWN, buff=0.15)

        origin_label = MathTex("O", font_size=24, color=WHITE).next_to(
            [0, 0, 0], DR, buff=0.1)

        # ---------- Build the static scene ----------
        self.play(FadeIn(lattice_points), Write(lattice_label))
        self.play(Create(ewald), FadeIn(center_dot), Write(center_label),
                  Write(origin_label))
        self.play(GrowArrow(k_in), Write(k_in_label))
        self.wait(0.5)

        bragg_text = Text(
            "Diffraction condition: a reciprocal point lies on the circle.",
            font_size=22, color=YELLOW
        ).to_edge(DOWN, buff=0.4)
        self.play(Write(bragg_text))
        self.wait(0.8)

        # ---------- Rotate the crystal ----------
        # Build a tracker for rotation and a group of "live" lattice points
        # that we can rotate around the origin.
        live_points = VGroup(*[
            Dot([h, k, 0], radius=DOT_RADIUS, color=BLUE)
            for h in range(-LATTICE_RANGE, LATTICE_RANGE + 1)
            for k in range(-LATTICE_RANGE, LATTICE_RANGE + 1)
            if not (h == 0 and k == 0)
        ])

        # Replace the static lattice points with live ones (origin stays fixed)
        self.remove(lattice_points)
        self.add(live_points)
        # Re-add the origin point on top
        origin_dot = Dot(ORIGIN, radius=DOT_RADIUS * 1.3, color=WHITE)
        self.add(origin_dot, origin_label)

        # Highlights — outgoing wavevector arrow + glow when a point hits the circle
        k_out = always_redraw(lambda: Arrow(
            center_C, _nearest_on_circle_point(live_points, center_C, R),
            color=RED, buff=0, stroke_width=4,
            max_tip_length_to_length_ratio=0.08,
        ))
        k_out_label = MathTex(r"\vec{k}_{\rm out}", font_size=28,
                              color=RED).next_to(k_in_label, RIGHT, buff=0.5)

        diffraction_dot = always_redraw(lambda: Dot(
            _nearest_on_circle_point(live_points, center_C, R),
            radius=0.16, color=RED).set_opacity(
                _proximity_opacity(live_points, center_C, R)
            )
        )

        self.add(k_out, k_out_label, diffraction_dot)
        self.play(Rotate(live_points, angle=ROTATIONS, about_point=ORIGIN),
                  run_time=RUN_TIME, rate_func=linear)
        self.wait(0.5)


def _nearest_on_circle_point(group, center, radius):
    """Return the (3D) coordinate of the lattice point closest to lying on
    the Ewald circle. Used to visualise k_out as it sweeps."""
    best = None
    best_err = 1e9
    cx, cy, _ = center
    for d in group:
        x, y, _ = d.get_center()
        err = abs(np.hypot(x - cx, y - cy) - radius)
        if err < best_err:
            best_err = err
            best = d.get_center().copy()
    return best if best is not None else np.array([0, 0, 0])


def _proximity_opacity(group, center, radius, sigma=0.1):
    """Opacity that peaks (1.0) when *any* lattice point sits on the Ewald
    circle and falls off otherwise. Drives the red 'diffraction!' flash."""
    cx, cy, _ = center
    best_err = 1e9
    for d in group:
        x, y, _ = d.get_center()
        err = abs(np.hypot(x - cx, y - cy) - radius)
        best_err = min(best_err, err)
    return float(np.exp(-(best_err / sigma) ** 2))

### The rendered animation

<video controls width="600" muted loop>
  <source src="../manim_demos/videos/02_ewald.mp4" type="video/mp4">
  Your viewer does not support embedded video. Open the file at
  <code>../manim_demos/videos/02_ewald.mp4</code> directly.
</video>

*Ewald construction: as the crystal rotates, reciprocal lattice points cross the Ewald circle and that reflection diffracts (red flash + outgoing wavevector).*

**The "aha":** static diagrams have to *tell* you "diffraction happens
when the point is on the sphere". The animation lets you *see* it happen,
repeatedly, as the lattice sweeps. Once that visual is in your head, the
3D version (rotating a Laue camera, or the precession method for protein
crystallography) makes immediate sense.

---
## Example 3 — BaTiO3 Cubic → Tetragonal Phase Transition

A barium-titanate sample cooled below 120 °C goes from cubic perovskite
($a=b=c$) to tetragonal ($a=b<c$). The corresponding XRD signature: the
single $(200)$ Bragg peak in the cubic phase splits into a $(200)$ and
$(002)$ doublet in the tetragonal phase, because the two interplanar
spacings are no longer equal.

This animation shows both panels at once: the unit cell distorting on the
left, the diffraction pattern splitting on the right, both driven by the
*same* underlying $c/a$ ratio.

### The prompt

> Write a Manim CE scene called `PhaseTransition` that animates the
> cubic→tetragonal phase transition in BaTiO3 with a side panel showing
> the simultaneous (200)/(002) XRD peak splitting.
>
> Left panel: a 2D top-down view of the perovskite unit cell with Ba
> atoms at the corners (blue) and Ti at the center (yellow). Use a
> ValueTracker for $c/a$ ratio, animate from 1.0 (cubic) to the room-
> temperature tetragonal value (1.011, exaggerated 5× for visibility).
> Show live $a$, $c$, $c/a$ readouts using the *true* lattice constants
> ($a = 4.000$ Å, $c_{\rm tet} = 4.040$ Å).
>
> Right panel: a $2\theta$ axis from 44° to 46°, showing the (200) and
> (002) peaks computed by Bragg's law from the current $a$ and $c$ at
> Cu Kα wavelength ($\lambda = 1.5406$ Å). Use Lorentzian peak shapes
> with FWHM 0.06°. Label the peaks live as they separate.
>
> Caption the phases: "Cubic phase (T > 120 °C): one peak", then
> "Cooling through Tc: cell elongates, peak splits", then "Tetragonal
> phase (T < 120 °C): (200) and (002) split".

### Claude's code

In [ ]:
"""Cubic -> tetragonal phase transition in BaTiO3, with the simultaneous
(200) -> (200)/(002) XRD peak splitting shown in a side panel.

Left panel:
  Simplified 2D view of the perovskite unit cell.
  Ba at corners, Ti at center, O atoms omitted for clarity.
  As we cool through the Curie temperature (~120 C), the cubic cell
  (a = b = c) elongates along c (a = b < c).

Right panel:
  Zoomed 2theta axis (44 - 46 deg) showing the (200) and (002) Bragg peaks.
  Cu K-alpha (lambda = 1.5406 A); peak position 2theta = 2 arcsin(lambda / 2d).
  In the cubic phase (a = c = 4.00 A) we see one peak at ~45.1 deg.
  In the room-temperature tetragonal phase (a = 3.99 A, c = 4.036 A) the
  (200) shifts slightly higher in 2theta and the (002) shifts lower.

For visual clarity the unit-cell distortion is shown enlarged
(scale factor 5x). The peak positions on the right are computed at the
*true* lattice parameters at every frame.
"""
from manim import *
import numpy as np


LAMBDA = 1.5406            # Cu K-alpha, angstrom
A_LATTICE = 4.000          # cubic (and tetragonal a-axis), angstrom
C_LATTICE_TET = 4.040      # tetragonal c-axis at room temp, angstrom
DISTORTION_VISUAL_GAIN = 5.0  # exaggerate cell elongation 5x for visibility


def two_theta(d_angstrom: float) -> float:
    """Bragg's law in degrees, n=1, Cu Kalpha."""
    return 2.0 * np.degrees(np.arcsin(LAMBDA / (2.0 * d_angstrom)))


class PhaseTransition(Scene):
    def construct(self):
        # ---------- Title ----------
        title = Text(
            "BaTiO3: cubic -> tetragonal phase transition",
            font_size=28,
        ).to_edge(UP, buff=0.3)
        subtitle = Text(
            "Cell distortion (left) shifts the (200) Bragg peak (right)",
            font_size=18, color=GREY,
        ).next_to(title, DOWN, buff=0.1)
        self.play(Write(title), Write(subtitle))

        # ---------- Left panel: unit cell ----------
        # ValueTracker: t in [0, 1]; t=0 cubic, t=1 fully tetragonal.
        t = ValueTracker(0.0)

        cell_center = LEFT * 3.5
        a_visual = 1.5      # base side length in scene units

        def cell_size():
            tt = t.get_value()
            c_over_a = 1.0 + tt * (C_LATTICE_TET / A_LATTICE - 1.0) * DISTORTION_VISUAL_GAIN
            return a_visual, a_visual * c_over_a

        def cell_corners():
            w, h = cell_size()
            return [
                cell_center + np.array([-w / 2, -h / 2, 0]),
                cell_center + np.array([+w / 2, -h / 2, 0]),
                cell_center + np.array([+w / 2, +h / 2, 0]),
                cell_center + np.array([-w / 2, +h / 2, 0]),
            ]

        # Cell outline
        cell_box = always_redraw(
            lambda: Polygon(*cell_corners(), color=WHITE, stroke_width=2)
        )

        # Ba at corners (large, blue), Ti at center (smaller, yellow)
        ba_atoms = always_redraw(lambda: VGroup(*[
            Dot(pt, radius=0.16, color=BLUE) for pt in cell_corners()
        ]))
        ti_atom = always_redraw(lambda: Dot(cell_center, radius=0.10, color=YELLOW))

        # Atom legend
        ba_legend = VGroup(
            Dot(radius=0.10, color=BLUE),
            Text("Ba", font_size=18),
        ).arrange(RIGHT, buff=0.15).next_to(cell_center, DOWN * 3 + LEFT * 0.7)
        ti_legend = VGroup(
            Dot(radius=0.07, color=YELLOW),
            Text("Ti", font_size=18),
        ).arrange(RIGHT, buff=0.15).next_to(ba_legend, RIGHT, buff=0.5)
        legend_note = Text("(O atoms omitted for clarity)",
                           font_size=14, color=GREY).next_to(ba_legend, DOWN, buff=0.1)

        # Live readout of c/a
        ratio_label = always_redraw(lambda: MathTex(
            rf"c/a = {1.0 + t.get_value() * (C_LATTICE_TET / A_LATTICE - 1.0):.4f}",
            font_size=24,
        ).next_to(cell_center, UP * 2.3))

        # Live a, c values (real, not visual)
        ac_label = always_redraw(lambda: MathTex(
            rf"a = {A_LATTICE:.3f}\,\text{{\AA}},"
            rf"\quad c = {A_LATTICE + t.get_value() * (C_LATTICE_TET - A_LATTICE):.3f}\,\text{{\AA}}",
            font_size=20, color=GREY,
        ).next_to(ratio_label, UP, buff=0.2))

        self.play(Create(cell_box), FadeIn(ba_atoms), FadeIn(ti_atom),
                  FadeIn(ba_legend), FadeIn(ti_legend), FadeIn(legend_note),
                  Write(ratio_label), Write(ac_label))

        # ---------- Right panel: Bragg peaks ----------
        axes_origin = RIGHT * 2.5 + DOWN * 0.5
        axes = Axes(
            x_range=[44, 46, 0.5],
            y_range=[0, 1.2, 0.5],
            x_length=4.5,
            y_length=3.5,
            tips=False,
            axis_config={"color": GREY, "stroke_width": 2,
                         "include_numbers": False},
        ).move_to(axes_origin)
        x_label = MathTex(r"2\theta\,(\text{deg})", font_size=22).next_to(
            axes.x_axis, DOWN, buff=0.2)
        y_label = MathTex(r"I", font_size=22).next_to(axes.y_axis, LEFT, buff=0.2)

        # x-axis tick labels
        x_ticks = VGroup(*[
            MathTex(f"{x:.0f}", font_size=18).next_to(axes.c2p(x, 0), DOWN, buff=0.1)
            for x in [44, 45, 46]
        ])

        self.play(Create(axes), Write(x_label), Write(y_label), Write(x_ticks))

        # Live diffraction pattern: sum of two Lorentzians at (200) and (002) positions
        FWHM = 0.06   # peak width in 2theta degrees

        def pattern_curve():
            tt = t.get_value()
            a = A_LATTICE
            c = a + tt * (C_LATTICE_TET - a)
            two_theta_200 = two_theta(a / 2.0)   # d_(200) = a/2
            two_theta_002 = two_theta(c / 2.0)   # d_(002) = c/2

            def lorentz(x, x0, w):
                return (w / 2)**2 / ((x - x0)**2 + (w / 2)**2)

            def y(x):
                return min(1.0, lorentz(x, two_theta_200, FWHM) +
                                lorentz(x, two_theta_002, FWHM))

            return axes.plot(y, x_range=[44, 46, 0.005], color=ORANGE,
                             stroke_width=3)

        pattern = always_redraw(pattern_curve)

        # Peak labels (also live)
        def peak_labels():
            tt = t.get_value()
            a = A_LATTICE
            c = a + tt * (C_LATTICE_TET - a)
            tt200 = two_theta(a / 2.0)
            tt002 = two_theta(c / 2.0)
            split = abs(tt200 - tt002)
            if split < 0.05:
                return VGroup(
                    MathTex(r"(200)", font_size=20, color=ORANGE).next_to(
                        axes.c2p(tt200, 1.05), UP, buff=0.05)
                )
            return VGroup(
                MathTex(r"(200)", font_size=20, color=ORANGE).next_to(
                    axes.c2p(tt200, 1.05), UR, buff=0.05),
                MathTex(r"(002)", font_size=20, color=ORANGE).next_to(
                    axes.c2p(tt002, 1.05), UL, buff=0.05),
            )
        labels_live = always_redraw(peak_labels)

        self.play(Create(pattern), FadeIn(labels_live))
        self.wait(0.6)

        cubic_caption = Text("Cubic phase (T > 120 C):  one peak",
                             font_size=22, color=BLUE).to_edge(DOWN, buff=0.6)
        self.play(Write(cubic_caption))
        self.wait(1.0)

        # ---------- The transition itself ----------
        new_caption = Text("Cooling through Tc:  cell elongates, peak splits",
                           font_size=22, color=YELLOW).to_edge(DOWN, buff=0.6)
        self.play(Transform(cubic_caption, new_caption))
        self.play(t.animate.set_value(1.0), run_time=4.5, rate_func=smooth)
        self.wait(0.5)

        final_caption = Text("Tetragonal phase (T < 120 C):  (200) and (002) split",
                             font_size=22, color=ORANGE).to_edge(DOWN, buff=0.6)
        self.play(Transform(cubic_caption, final_caption))
        self.wait(2.0)

### The rendered animation

<video controls width="600" muted loop>
  <source src="../manim_demos/videos/03_phase_transition.mp4" type="video/mp4">
  Your viewer does not support embedded video. Open the file at
  <code>../manim_demos/videos/03_phase_transition.mp4</code> directly.
</video>

*BaTiO3 phase transition: a single (200) peak in the cubic phase splits into (200) + (002) as the cell elongates below the Curie temperature.*

**What's quietly correct:** the peak positions on the right are
computed from the *real* lattice constants at every frame using Bragg's
law with Cu Kα. Only the *visual* unit-cell distortion is exaggerated
(true $c/a = 1.011$ would be invisible at this scale; we show it as ~1.05).
The split between $(200)$ and $(002)$ is the actual ~0.5° split you'd
see on a real diffractometer at this peak.

---
## Example 4 — Ostwald Ripening

Many small particles in a fixed volume of solute don't stay that way: large
particles grow at the expense of small ones, because surface curvature
makes small particles' chemical potential higher. The classical model is
LSW (Lifshitz–Slyozov–Wagner) theory:

$$\frac{dr_i}{dt} = K\left(\frac{1}{\langle r\rangle} - \frac{1}{r_i}\right)$$

This animation pre-computes 60 timesteps of LSW dynamics for a population
of 30 particles and animates the result. It pairs naturally with **notebook
04** (where we built an agent that *measures* a particle-size distribution
from a microscopy image): now you see *why* that distribution looks the way
it does after some aging time.

### The prompt

> Write a Manim CE scene called `Ostwald` that animates Ostwald ripening
> of a population of spherical particles inside a square box. Pre-compute
> the LSW dynamics in numpy: 30 particles with initial radii drawn from
> N(0.18, 0.06), placed without overlap in a 6×6 box. Integrate
> $dr_i/dt = K(1/\langle r\rangle - 1/r_i)$ where $\langle r\rangle$
> is the volume-weighted mean radius. Use $K = 0.005$, $dt = 0.04$, 60
> steps. Particles whose radius drops below 0.04 dissolve. Enforce mass
> conservation by rescaling radii each step so total $\sum r_i^3$ is
> constant.
>
> Then animate: particles grow/shrink in place; dissolved ones fade out;
> a side panel shows a live histogram of radii using bins of width 0.0625;
> live readouts show the number of particles alive and the mean radius;
> title and equation pinned to the top.

### Claude's code

In [ ]:
"""Nucleation and Ostwald ripening.

Many small spherical particles in a fixed volume of solute. Each particle
exchanges atoms with the solute according to the Gibbs-Thomson driving
force: small particles have higher chemical potential (mu ~ 1/r), so they
shed atoms; large particles capture them. The result, observed by Ostwald
in 1896, is that the size distribution shifts to the right and the number
of particles decreases — *coarsening*.

The simplest quantitative model is LSW theory (Lifshitz-Slyozov-Wagner):
    dr_i/dt = K * (1/<r> - 1/r_i)
where <r> is the critical radius (here approximated by the volume-weighted
mean). Particles whose radius drops to zero dissolve.

We integrate this on a small population of ~30 particles and animate the
result with Manim. Mass is conserved by construction: the rate of total
volume change is zero on average over LSW dynamics, but for a finite
population we apply a tiny rescaling each step to keep total volume exact.
"""
from manim import *
import numpy as np


N_PARTICLES = 30
BOX_HALF = 3.0          # half-size of the simulation box (scene units)
R_INITIAL_MEAN = 0.18   # initial mean radius
R_INITIAL_STD = 0.06
N_FRAMES = 60           # number of timesteps to precompute
DT = 0.04               # timestep
K_LSW = 0.005           # diffusion-limited rate constant
SEED = 1


def simulate(seed=SEED):
    """Pre-compute particle positions and radii over time.

    Returns: list of (positions[N,2], radii[N]) snapshots, one per frame.
    Particles that dissolve get radius 0 (kept in the array for indexing
    stability; the renderer hides them)."""
    rng = np.random.default_rng(seed)

    # ---- Initial positions (rejection sampling for non-overlap) ----
    radii = np.clip(
        rng.normal(R_INITIAL_MEAN, R_INITIAL_STD, size=N_PARTICLES),
        0.06, 0.35,
    )
    positions = np.zeros((N_PARTICLES, 2))
    for i in range(N_PARTICLES):
        for _ in range(2000):
            xy = rng.uniform(-BOX_HALF + radii[i] + 0.05,
                              BOX_HALF - radii[i] - 0.05, size=2)
            ok = True
            for j in range(i):
                if radii[j] == 0:
                    continue
                if np.hypot(*(xy - positions[j])) < radii[i] + radii[j] + 0.05:
                    ok = False; break
            if ok:
                positions[i] = xy
                break

    snapshots = [(positions.copy(), radii.copy())]
    initial_volume = np.sum(radii ** 3)

    for _ in range(N_FRAMES):
        alive = radii > 0
        if alive.sum() == 0:
            break
        # Volume-weighted mean radius is the natural critical radius
        r_alive = radii[alive]
        r_crit = np.sum(r_alive ** 3) / np.sum(r_alive ** 2)

        dr = np.zeros_like(radii)
        dr[alive] = K_LSW * (1.0 / r_crit - 1.0 / r_alive)

        radii = radii + DT * dr
        radii = np.where(radii < 0.04, 0.0, radii)

        # Enforce mass conservation by tiny rescaling (numerical safety)
        cur_vol = np.sum(radii ** 3)
        if cur_vol > 0:
            radii = radii * (initial_volume / cur_vol) ** (1 / 3)

        snapshots.append((positions.copy(), radii.copy()))

    return snapshots


SNAPSHOTS = simulate()


class Ostwald(Scene):
    def construct(self):
        # ---------- Title ----------
        title = Text("Ostwald ripening: large particles eat small ones",
                     font_size=26).to_edge(UP, buff=0.3)
        subtitle = MathTex(
            r"\frac{dr_i}{dt} = K\left(\frac{1}{\langle r\rangle} - \frac{1}{r_i}\right)"
            r"\quad \text{(LSW theory)}",
            font_size=22, color=GREY,
        ).next_to(title, DOWN, buff=0.15)
        self.play(Write(title), Write(subtitle))

        # ---------- Box outline ----------
        box = Square(side_length=2 * BOX_HALF, color=GREY,
                     stroke_width=2).shift(LEFT * 2.5)
        box_offset = LEFT * 2.5
        self.play(Create(box))

        # ---------- Particles ----------
        positions, radii = SNAPSHOTS[0]
        circles = []
        for i in range(N_PARTICLES):
            c = Circle(radius=radii[i], color=BLUE_E, fill_opacity=0.7,
                       stroke_color=BLUE_A, stroke_width=1.5).move_to(
                np.array([positions[i, 0], positions[i, 1], 0]) + box_offset)
            circles.append(c)
        circle_group = VGroup(*circles)
        self.play(FadeIn(circle_group))

        # ---------- Side panel: histogram of radii ----------
        hist_axes = Axes(
            x_range=[0, 0.5, 0.1],
            y_range=[0, 12, 4],
            x_length=4.2,
            y_length=3.2,
            tips=False,
            axis_config={"color": GREY, "stroke_width": 2,
                         "include_numbers": False},
        ).shift(RIGHT * 3.5 + DOWN * 0.3)
        hist_x_label = MathTex(r"r", font_size=20).next_to(hist_axes.x_axis, DOWN)
        hist_y_label = MathTex(r"\#\,\text{particles}", font_size=18).next_to(
            hist_axes.y_axis, LEFT)
        hist_title = Text("Size distribution", font_size=18).next_to(
            hist_axes, UP, buff=0.1)
        self.play(Create(hist_axes), Write(hist_x_label),
                  Write(hist_y_label), Write(hist_title))

        bin_edges = np.linspace(0, 0.5, 9)

        def histogram_bars(snap_radii):
            alive = snap_radii[snap_radii > 0]
            counts, _ = np.histogram(alive, bins=bin_edges)
            bars = VGroup()
            for cnt, lo, hi in zip(counts, bin_edges[:-1], bin_edges[1:]):
                if cnt == 0:
                    continue
                bottom_left  = hist_axes.c2p(lo, 0)
                bottom_right = hist_axes.c2p(hi, 0)
                top_left     = hist_axes.c2p(lo, cnt)
                top_right    = hist_axes.c2p(hi, cnt)
                bar = Polygon(bottom_left, bottom_right, top_right, top_left,
                              color=BLUE_E, fill_opacity=0.6, stroke_width=1)
                bars.add(bar)
            return bars

        bars = histogram_bars(SNAPSHOTS[0][1])
        self.add(bars)

        # ---------- Live readout ----------
        readout_pos = box.get_corner(DR) + DOWN * 0.4
        readout = always_redraw(lambda: VGroup(
            Text(f"# particles alive: {int(np.sum(SNAPSHOTS[CURRENT_FRAME[0]][1] > 0))}",
                 font_size=18),
            Text(f"mean radius: {np.mean(SNAPSHOTS[CURRENT_FRAME[0]][1][SNAPSHOTS[CURRENT_FRAME[0]][1] > 0]):.3f}",
                 font_size=18),
        ).arrange(DOWN, aligned_edge=LEFT, buff=0.1).move_to(readout_pos))
        self.add(readout)

        # ---------- Animate the timesteps ----------
        # We step through snapshots, animating radius changes only (positions
        # are fixed). At each step we also rebuild the histogram bars.
        for frame_idx in range(1, len(SNAPSHOTS)):
            CURRENT_FRAME[0] = frame_idx
            new_radii = SNAPSHOTS[frame_idx][1]

            anims = []
            for i, c in enumerate(circles):
                if new_radii[i] > 0:
                    target = Circle(radius=new_radii[i], color=BLUE_E,
                                    fill_opacity=0.7, stroke_color=BLUE_A,
                                    stroke_width=1.5).move_to(c.get_center())
                    anims.append(Transform(c, target))
                else:
                    anims.append(c.animate.set_opacity(0))

            new_bars = histogram_bars(new_radii)
            anims.append(Transform(bars, new_bars))

            self.play(*anims, run_time=0.08, rate_func=linear)

        self.wait(1.5)


# Mutable holder so always_redraw lambdas can reference current frame index
CURRENT_FRAME = [0]

### The rendered animation

<video controls width="600" muted loop>
  <source src="../manim_demos/videos/04_ostwald.mp4" type="video/mp4">
  Your viewer does not support embedded video. Open the file at
  <code>../manim_demos/videos/04_ostwald.mp4</code> directly.
</video>

*Ostwald ripening: LSW dynamics on 30 particles. Small particles dissolve; large ones grow; the histogram shifts to the right as the size distribution coarsens.*

**The thing to notice:** the histogram doesn't just shift — it also
narrows in shape. That's the classical LSW result: in the long-time limit
the size distribution approaches a self-similar shape with a characteristic
right-skew. Real microscopy data on aged samples follows this distribution
shape closely.

---
## What this is good for

| Use case | Why AI + Manim helps |
|---|---|
| **Teaching** — making intuitive figures for a course or lecture | Animated geometry is dramatically clearer than the same static figure; you describe the concept, the model writes the code |
| **Talks** — illustrating a result in a conference presentation | Drop a 10-second MP4 into a slide instead of a still image; embed the source `.py` for reproducibility |
| **Papers** — supplementary videos | Build a polished SI animation in 30 minutes instead of a day |
| **Onboarding** — explaining your group's technique to a new student | A 30-second animation of *your* method (ptychography, fluorescence tomography, etc.) is a great first conversation |
| **Outreach** — making your science visible | Same code, render at `-qh` (1080p) for social media or YouTube |

## What it isn't good for

- **Quantitative figures** — Manim is for *intuition*, not for plotting
  data. Use matplotlib for that.
- **Real-time interaction** — Manim renders to MP4. For interactive
  animations use plotly, bokeh, or NiceGUI.
- **3D crystallography** — possible but heavy. For real molecular
  visualization use VMD, OVITO, or Jmol.

## Iteration tips

- Start with `-ql` (low quality, 480p, ~5 sec/scene). Only render at higher
  quality once the content is right.
- Manim's strict-mode error messages are surprisingly readable. Paste them
  back to Claude with the scene code and it usually finds the bug in one
  iteration.
- For complicated scenes, ask Claude to **first describe the layout in
  words** before writing code. You'll catch design errors faster.
- Long scenes are slow to iterate. Build each animation as several short
  scenes and stitch them with ffmpeg if you need a single file.

## Where to read more

- Manim CE docs: <https://docs.manim.community>
- Tutorial example gallery: <https://docs.manim.community/en/stable/examples.html>
- 3Blue1Brown's original Manim talk: <https://www.youtube.com/watch?v=ENMyFGmq5OA>